# Frozen 20-Run PPO CUDA Training

This notebook resumes the frozen validation-selected PPO inventory. It does not evaluate the held-out test cohort.

> Research-only simulation. Not a medical device and not for clinical dosing or patient care.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

repo = Path('/content/VitalDB-Feature-Selection')
remote = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
if not repo.exists():
    subprocess.run(['git', 'clone', remote, str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(repo), 'reset', '--hard', 'origin/main'], check=True)
required = '767f3bff3dcaeabc51049fba5ccba1ac02b69ae3'
subprocess.run(['git', '-C', str(repo), 'merge-base', '--is-ancestor', required, 'HEAD'], check=True)
os.chdir(repo)
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
importlib.invalidate_caches()
print('Repository commit:', subprocess.run(
    ['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip())

In [ ]:
import pandas as pd
import torch

torch_before = torch.__version__
pandas_before = pd.__version__
dry_run = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--dry-run', '-r', 'requirements-rl.txt'],
    check=True, capture_output=True, text=True
)
proposal = dry_run.stdout + dry_run.stderr
if 'Would install torch-' in proposal or 'Would install pandas-' in proposal:
    raise RuntimeError('RL dependency profile would replace torch or pandas; aborting.')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', 'stable-baselines3==2.9.0'
], check=True)
import stable_baselines3
import pandas as pd_after
import torch as torch_after
if torch_after.__version__ != torch_before or pd_after.__version__ != pandas_before:
    raise RuntimeError('torch/pandas changed during RL dependency setup.')
if not torch_after.cuda.is_available():
    raise RuntimeError('Full PPO notebook requires a CUDA runtime.')
print('SB3:', stable_baselines3.__version__)
print('Torch/CUDA preserved:', torch_after.__version__, torch_after.cuda.get_device_name(0))
print('Pandas preserved:', pd_after.__version__)

In [ ]:
drive_root = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
protocol_dir = drive_root / 'outputs/ppo_protocol'
output_root = drive_root / 'outputs/ppo_control_comparison'
protocol_dir.mkdir(parents=True, exist_ok=True)
output_root.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, 'scripts/run_ppo_experiment.py',
    '--dataset-dir', str(repo / 'data/modeling/full'),
    '--protocol-dir', str(protocol_dir),
    '--output-root', str(output_root),
    '--initialize-only',
], check=True)
protocol = json.loads((protocol_dir / 'frozen_ppo_protocol.json').read_text())
print(json.dumps({
    'protocol_hash': protocol['protocol_hash'],
    'inventory': protocol['inventory'],
    'cohort': protocol['cohort'],
    'action': protocol['action'],
    'reward': protocol['reward'],
    'ppo': protocol['ppo'],
    'confirmation_text': protocol['confirmation_text'],
    'test_cohort_accessed': False,
}, indent=2))

In [ ]:
expected = protocol['confirmation_text']
confirmation = input(f'Type {expected} to start or resume the frozen CUDA inventory: ').strip()
if confirmation != expected:
    raise RuntimeError('Confirmation mismatch; full training remains locked.')
subprocess.run([
    sys.executable, 'scripts/run_ppo_experiment.py',
    '--dataset-dir', str(repo / 'data/modeling/full'),
    '--protocol-dir', str(protocol_dir),
    '--output-root', str(output_root),
    '--device', 'cuda',
    '--confirmation', confirmation,
], check=True)

Completed runs are skipped and partial runs resume from `last_model.zip`. Validation selects checkpoints; the test cohort remains sealed.